In [3]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

usn_programs = {
    "Bachelor i ingeniørfag, dataingeniør": "#/studieplan/ING2_2025_H%C3%98ST",
}

rows = []

async def scrape_usn():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for study_program, hash_fragment in usn_programs.items():
            print(f"\n🔍 Extracting for: {study_program}")
            await page.goto("https://www.usn.no/studier/studie-og-emneplaner/")
            await page.wait_for_timeout(2000)
            await page.evaluate(f"window.location.hash = '{hash_fragment}'")
            await page.wait_for_timeout(4000)

            # Accept cookies
            try:
                await page.get_by_text("Godta alle").click(timeout=3000)
                print("🍪 Accepted cookies")
            except:
                print("⚠️ No cookie popup")

            await page.wait_for_selector("usn-study-model")

            print("🔍 Trying to access shadow DOM headers...")
            try:
                # Log all header texts inside shadow DOM
                headers = await page.evaluate('''
                    () => {
                        const root = document.querySelector('usn-study-model')?.shadowRoot;
                        return [...(root?.querySelectorAll('div.header') || [])].map(e => e.textContent.trim());
                    }
                ''')
                print("🧩 Headers found inside shadow DOM:")
                for h in headers:
                    print("   -", h)

                # Helper: Click a shadow header
                async def click_header_with_text(text):
                    clicked = await page.evaluate(f'''
                        () => {{
                            const root = document.querySelector('usn-study-model')?.shadowRoot;
                            const headers = [...(root?.querySelectorAll('div.header') || [])];
                            const match = headers.find(h => h.textContent.includes("{text}"));
                            match?.click();
                            return !!match;
                        }}
                    ''')
                    if clicked:
                        print(f"✅ Clicked header: '{text}'")
                    else:
                        print(f"❌ Could not find header: '{text}'")
                    return clicked

                # Navigate through structure
                if "ingeniørfag" in study_program.lower():
                    if not await click_header_with_text("Dataingeniør"):
                        raise Exception("Missing: 'Dataingeniør'")
                    await page.wait_for_timeout(1000)

                    if not await click_header_with_text("Cyber physical systems Kongsberg A-vei"):
                        raise Exception("Missing: specialization")
                    await page.wait_for_timeout(1000)

                    if not await click_header_with_text("Obligatoriske emner"):
                        raise Exception("Missing: 'Obligatoriske emner'")
                    await page.wait_for_timeout(1000)

                # Scrape
                html = await page.evaluate('''
                    () => document.querySelector('usn-study-model')?.shadowRoot?.innerHTML
                ''')

                soup = BeautifulSoup(html, "html.parser")
                found_any = False
                for a in soup.select("a.studiemodell"):
                    if "Obligatorisk" in a.get("data-original-title", ""):
                        rows.append({
                            "school": "USN",
                            "study_program": study_program,
                            "type": "Obligatorise_emner",
                            "mandatory_subject": a.text.strip()
                        })
                        found_any = True

                if found_any:
                    print(f"✅ Found subjects for {study_program}")
                else:
                    print("⚠️ No mandatory subjects found.")
                    rows.append({
                        "school": "USN",
                        "study_program": study_program,
                        "type": "Obligatorise_emner",
                        "mandatory_subject": ""
                    })

            except Exception as e:
                print(f"❌ Error during scraping of '{study_program}': {e}")
                rows.append({
                    "school": "USN",
                    "study_program": study_program,
                    "type": "Obligatorise_emner",
                    "mandatory_subject": ""
                })

        await browser.close()

# Run it
await scrape_usn()

# Save
df = pd.DataFrame(rows, columns=["school", "study_program", "type", "mandatory_subject"])
df.to_csv("Mandatory_subjects_playwright.csv", index=False)
print("\n📄 Finished writing USN mandatory subjects.")



🔍 Extracting for: Bachelor i ingeniørfag, dataingeniør
⚠️ No cookie popup
🔍 Trying to access shadow DOM headers...
🧩 Headers found inside shadow DOM:
❌ Could not find header: 'Dataingeniør'
❌ Error during scraping of 'Bachelor i ingeniørfag, dataingeniør': Missing: 'Dataingeniør'

📄 Finished writing USN mandatory subjects.
